# Mixture of Experts (MoE) 教程

本教程介绍 MoE 稀疏专家混合模型的核心概念和实现。

## 目录
1. MoE 基础概念
2. 路由器机制
3. 专家模块
4. 完整 MoE 模型

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

from moe import (
    MoEConfig, RouterType, Expert,
    TopKRouter, SwitchRouter, ExpertChoiceRouter,
    MoELayer, MoEModel, create_moe_model,
    softmax, compute_load_balancing_loss,
)

## 1. MoE 基础概念

MoE 通过稀疏激活实现条件计算：每个 token 只激活部分专家。

In [ ]:
# 配置示例
config = MoEConfig(
    d_model=64,
    n_experts=8,
    n_experts_per_tok=2,
    d_ff=256,
)

print(f"模型维度: {config.d_model}")
print(f"专家数量: {config.n_experts}")
print(f"每 token 激活专家数: {config.n_experts_per_tok}")
print(f"稀疏度: {config.n_experts_per_tok / config.n_experts * 100:.1f}%")

## 2. 路由器机制

In [ ]:
# Top-K 路由器
router = TopKRouter(d_model=64, n_experts=8, n_experts_per_tok=2)
x = np.random.randn(20, 64)  # 20 tokens

indices, weights, aux = router(x)
print(f"选中专家索引: {indices[:5]}")
print(f"专家权重: {weights[:5]}")
print(f"负载均衡损失: {aux['aux_loss']:.4f}")

In [ ]:
# 可视化专家分布
plt.figure(figsize=(10, 4))
plt.bar(range(8), aux['expert_counts'])
plt.xlabel('Expert ID')
plt.ylabel('Token Count')
plt.title('Expert Load Distribution')
plt.show()

## 3. 专家模块

In [ ]:
# 单个专家
expert = Expert(d_model=64, d_ff=256)
x = np.random.randn(10, 64)
y = expert(x)
print(f"Input: {x.shape} -> Output: {y.shape}")

## 4. 完整 MoE 模型

In [ ]:
# 创建 MoE 模型
config = MoEConfig(
    d_model=64, n_layers=2, n_experts=4,
    n_heads=4, d_ff=128, vocab_size=1000
)
model = MoEModel(config)

# 前向传播
input_ids = np.random.randint(0, 1000, size=(2, 10))
labels = np.random.randint(0, 1000, size=(2, 10))
result = model(input_ids, labels=labels)

print(f"Logits: {result['logits'].shape}")
print(f"LM Loss: {result['lm_loss']:.4f}")
print(f"Aux Loss: {result['aux_loss']:.4f}")
print(f"Total Loss: {result['loss']:.4f}")

In [ ]:
# 使用工厂函数
model = create_moe_model('small', n_experts=8)
print(f"Model: d_model={model.config.d_model}, experts={model.config.n_experts}")

## 总结

MoE 核心优势:
1. **稀疏激活**: 每个 token 只激活 K 个专家
2. **扩展容量**: 参数量增加但计算量不变
3. **负载均衡**: 辅助损失确保专家均匀利用